# The Standard Model in FeynPy (`SM.fr` style)

This notebook is the FeynPy analogue of the FeynRules model file `SM.fr`.


## Setup

In [1]:
import re
import sys
from fractions import Fraction
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (ROOT, SRC):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from symbolica import Expression, S

from feynpy import (
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    CovD,
    Field,
    FieldStrength,
    FieldTransformation,
    Gamma,
    GaugeFixing,
    GaugeGroup,
    GaugeRepresentation,
    GhostLagrangian,
    LORENTZ_INDEX,
    Model,
    Parameter,
    PartialD,
    ProjM,
    ProjP,
    SPINOR_INDEX,
    WEAK_ADJ_INDEX,
    WEAK_FUND_INDEX,
    flavor_index,
    rotation,
)
from symbolic.spenso_structures import (
    gauge_generator,
    structure_constant,
    weak_eps2,
    weak_gauge_generator,
    weak_structure_constant,
)
from symbolic.vertex_engine import I
from symbolic.tensor_canonicalization import canonize_full
from lagrangian.symbolica_export import interaction_term_to_symbolica

num = Expression.num
HALF = num(1) / num(2)
INV_SQRT2 = HALF ** HALF

ANSI = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")
DISPLAY_PREFIX = re.compile(r"(?:python|spenso(?:_python)?)::(?:\{[^}]*\}::)?")
DISPLAY_TAG = re.compile(r"\{[^}]*\}::")


def clean(text):
    if hasattr(text, "to_symbolica") and not hasattr(text, "to_canonical_string"):
        text = text.to_symbolica()
    rendered = text.to_canonical_string() if hasattr(text, "to_canonical_string") else str(text)
    rendered = ANSI.sub("", rendered)
    rendered = DISPLAY_PREFIX.sub("", rendered)
    return DISPLAY_TAG.sub("", rendered)


def show(title, result):
    print("==========")
    print(title)
    print(clean(result))
    print()

def show_compiled(title, lagrangian):
    print("==========")
    print(title)
    for term in lagrangian.terms:
        print(clean(interaction_term_to_symbolica(term)))
    print()

def show_model(title, result):
    print("==========")
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", clean(expression))
            print()
    else:
        print(clean(result))
        print()


FeynmanGauge = True
ModelName = "Standard Model (physical basis)"

## 2. Indices

`SM.fr` declares `Generation`, `SU2W` (weak adjoint), `SU2D` (weak doublet),
`Colour`, and `Gluon`. The colour, weak and Lorentz/spinor index types ship with
FeynPy; only the flavour index has to be declared, exactly like in
`list_lagrangians.ipynb`.

In [2]:
Generation = flavor_index("Generation", 3, prefix="fl")
Generation

IndexType(name='Generation', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='generation', dimension=3, is_flavor=True, prefix='fl')

## 3. Parameters

The `M$Parameters` block: gauge couplings `g1, g2, g3`, the Higgs `vev` and
quartic `lam`, the derived electroweak quantities `sw, cw, ee`, the masses, the
Yukawa matrices and the (Cabibbo-only) `CKM` matrix.

`Parameter(..., value=...)` mirrors a FeynRules `Internal` parameter
(`Definitions`), while `Parameter(..., indices=..., components=...)` mirrors an
indexed parameter with an explicit `Value` table.

In [3]:
g1 = Parameter("g1")
g2 = Parameter("g2")
g3 = Parameter("g3")
lam = Parameter("lam")
vev = Parameter("vev")

gz = (g1.symbol**2 + g2.symbol**2) ** HALF
sw = Parameter("sw", value=g1.symbol / gz)            # sin(theta_W) = g1 / sqrt(g1^2+g2^2)
cw = Parameter("cw", value=g2.symbol / gz)            # cos(theta_W) = g2 / sqrt(g1^2+g2^2)
ee = Parameter("ee", value=g1.symbol * g2.symbol / gz)  # e = g1 g2 / sqrt(g1^2+g2^2)

MW = Parameter("MW", value=g2.symbol * vev.symbol / 2)
MZ = Parameter("MZ", value=gz * vev.symbol / 2)
MH = Parameter("MH", value=(2 * lam.symbol * vev.symbol**2) ** HALF)

# R_xi gauge parameters (kept symbolic)
xiA = Parameter("xiA", internal=False, value=S("xiA"))
xiZ = Parameter("xiZ", internal=False, value=S("xiZ"))
xiW = Parameter("xiW", internal=False, value=S("xiW"))
xiG = Parameter("xiG", internal=False, value=S("xiG"))

cabi = S("cabi")
cos, sin = S("cos")(cabi), S("sin")(cabi)


def diagonal(prefix):
    """Diagonal 3x3 Yukawa table, e.g. yu[i,i] -> yu1, yu2, yu3."""
    return {(r, c): (S(f"{prefix}{r}") if r == c else num(0))
            for r in range(1, 4) for c in range(1, 4)}


def mass_param(name, prefix):
    """Diagonal mass M[i] = vev * y[i] / sqrt(2)."""
    return Parameter(name, indices=(Generation,),
                     components={(i,): INV_SQRT2 * vev.symbol * S(f"{prefix}{i}") for i in range(1, 4)})


ckm = {(1, 1): cos, (1, 2): sin, (1, 3): num(0),
       (2, 1): -sin, (2, 2): cos, (2, 3): num(0),
       (3, 1): num(0), (3, 2): num(0), (3, 3): num(1)}
ckm_T = {(c, r): v for (r, c), v in ckm.items()}

Mvl = Parameter("Mvl", indices=(Generation,), components={(i,): num(0) for i in range(1, 4)})
Ml = mass_param("Ml", "ye")
Mu = mass_param("Mu", "yu")
Md = mass_param("Md", "yd")

Yu = Parameter("Yu", indices=(Generation, Generation), complex_param=True, components=diagonal("yu"))
YuDag = Parameter("YuDag", indices=(Generation, Generation), complex_param=True, components=diagonal("yu"))
Yd = Parameter("Yd", indices=(Generation, Generation), complex_param=True, components=diagonal("yd"))
YdDag = Parameter("YdDag", indices=(Generation, Generation), complex_param=True, components=diagonal("yd"))
Ye = Parameter("Ye", indices=(Generation, Generation), complex_param=True, components=diagonal("ye"))
YeDag = Parameter("YeDag", indices=(Generation, Generation), complex_param=True, components=diagonal("ye"))
CKM = Parameter("CKM", indices=(Generation, Generation), complex_param=True, components=ckm, unitary_partner="CKMDag")
CKMDag = Parameter("CKMDag", indices=(Generation, Generation), complex_param=True, components=ckm_T, unitary_partner="CKM")

all_parameters = (g1, g2, g3, lam, vev, sw, cw, ee, MW, MZ, MH, xiA, xiZ, xiW, xiG,
                  Mvl, Ml, Mu, Md, Yu, YuDag, Yd, YdDag, Ye, YeDag, CKM, CKMDag)
print("sin(theta_W) =", sw.value)
print("e            =", ee.value)
print("MW           =", MW.value)

sin(theta_W) = g1/(g1^2+g2^2)^(1/2)
e            = g1*g2/(g1^2+g2^2)^(1/2)
MW           = 1/2*g2*vev


## 4. Particle classes (field declarations)

- **Gauge-basis (unphysical) fields** `B, Wi, G, LL, lR, QL, uR, dR, Phi` and the
  gauge ghosts. These carry the gauge quantum numbers (`Y`, weak doublet index,
  colour) and are what the Lagrangian is written in.
- **Physical (mass-eigenstate) fields** `A, Z, W, H, G0, GP` and the fermion
  flavour classes `vl, l, uq, dq`.

In [4]:
# --- Gauge-basis (unphysical) fields ---
B = Field("B", spin=1, self_conjugate=True, indices=(LORENTZ_INDEX,))
Wi = Field("Wi", spin=1, self_conjugate=True, indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX))
G = Field("G", spin=1, self_conjugate=True, indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX))
ghB = Field("ghB", spin=0, kind="ghost", self_conjugate=False, ghost_of=B, 
            symbol=S("ghB"), conjugate_symbol=S("ghBbar"))
ghWi = Field("ghWi", spin=0, kind="ghost", self_conjugate=False, indices=(WEAK_ADJ_INDEX,), 
             ghost_of=Wi, symbol=S("ghWi"), conjugate_symbol=S("ghWibar"))
ghG = Field("ghG", spin=0, kind="ghost", self_conjugate=False, indices=(COLOR_ADJ_INDEX,), 
            ghost_of=G, symbol=S("ghG"), conjugate_symbol=S("ghGbar"))

LL = Field("LL", spin=Fraction(1, 2), self_conjugate=False,
           indices=(SPINOR_INDEX, WEAK_FUND_INDEX, Generation), quantum_numbers={"Y": -HALF})
lR = Field("lR", spin=Fraction(1, 2), self_conjugate=False,
           indices=(SPINOR_INDEX, Generation), quantum_numbers={"Y": -num(1)})
QL = Field("QL", spin=Fraction(1, 2), self_conjugate=False,
           indices=(SPINOR_INDEX, WEAK_FUND_INDEX, Generation, COLOR_FUND_INDEX),
           quantum_numbers={"Y": num(1) / num(6)})
uR = Field("uR", spin=Fraction(1, 2), self_conjugate=False,
           indices=(SPINOR_INDEX, Generation, COLOR_FUND_INDEX), quantum_numbers={"Y": num(2) / num(3)})
dR = Field("dR", spin=Fraction(1, 2), self_conjugate=False,
           indices=(SPINOR_INDEX, Generation, COLOR_FUND_INDEX), quantum_numbers={"Y": -(num(1) / num(3))})
Phi = Field("Phi", spin=0, self_conjugate=False, symbol=S("Phi"), conjugate_symbol=S("Phibar"),
            indices=(WEAK_FUND_INDEX,), quantum_numbers={"Y": HALF})

# --- Physical (mass-eigenstate) fields ---
W = Field("W", spin=1, self_conjugate=False, conjugate_symbol=S("Wbar"), indices=(LORENTZ_INDEX,),
          mass=MW.symbol, quantum_numbers={"Q": num(1)})
Z = Field("Z", spin=1, self_conjugate=True, indices=(LORENTZ_INDEX,), mass=MZ.symbol, quantum_numbers={"Q": num(0)})
A = Field("A", spin=1, self_conjugate=True, indices=(LORENTZ_INDEX,), mass=num(0), quantum_numbers={"Q": num(0)})
H = Field("H", spin=0, self_conjugate=True, mass=MH.symbol, quantum_numbers={"Q": num(0)})
G0 = Field("G0", spin=0, self_conjugate=True, mass=MZ.symbol, quantum_numbers={"Q": num(0)}, goldstone_of=Z)
GP = Field("GP", spin=0, self_conjugate=False, conjugate_symbol=S("GM"), mass=MW.symbol,
           quantum_numbers={"Q": num(1)}, goldstone_of=W)
ghA = Field(
    "ghA", spin=0, kind="ghost", self_conjugate=False,
    ghost_of=A, symbol=S("ghA"), conjugate_symbol=S("ghAbar"),
    mass=num(0), quantum_numbers={"GhostNumber": num(1)},
)
ghZ = Field(
    "ghZ", spin=0, kind="ghost", self_conjugate=False,
    ghost_of=Z, symbol=S("ghZ"), conjugate_symbol=S("ghZbar"),
    mass=MZ.symbol, quantum_numbers={"GhostNumber": num(1)},
)
ghWp = Field(
    "ghWp", spin=0, kind="ghost", self_conjugate=False,
    ghost_of=W, symbol=S("ghWp"), conjugate_symbol=S("ghWm"),
    mass=MW.symbol, quantum_numbers={"GhostNumber": num(1), "Q": num(1)},
)
ghWm = Field(
    "ghWm", spin=0, kind="ghost", self_conjugate=False,
    ghost_of=W, symbol=S("ghWm"), conjugate_symbol=S("ghWp"),
    mass=MW.symbol, quantum_numbers={"GhostNumber": num(1), "Q": -num(1)},
)

vl = Field("vl", spin=Fraction(1, 2), self_conjugate=False, indices=(SPINOR_INDEX, Generation),
           mass=Mvl, quantum_numbers={"Q": num(0), "LeptonNumber": num(1)},
           flavor_index=Generation, class_members=("ve", "vm", "vt"))
l = Field("l", spin=Fraction(1, 2), self_conjugate=False, indices=(SPINOR_INDEX, Generation),
          mass=Ml, quantum_numbers={"Q": -num(1), "LeptonNumber": num(1)},
          flavor_index=Generation, class_members=("e", "mu", "ta"))
uq = Field("uq", spin=Fraction(1, 2), self_conjugate=False, indices=(SPINOR_INDEX, Generation, COLOR_FUND_INDEX),
           mass=Mu, quantum_numbers={"Q": num(2) / num(3)}, flavor_index=Generation, class_members=("u", "c", "t"))
dq = Field("dq", spin=Fraction(1, 2), self_conjugate=False, indices=(SPINOR_INDEX, Generation, COLOR_FUND_INDEX),
           mass=Md, quantum_numbers={"Q": -(num(1) / num(3))}, flavor_index=Generation, class_members=("d", "s", "b"))

print("charged leptons:", [m.name for m in l.class_members])
print("up quarks:      ", [m.name for m in uq.class_members])
print("down quarks:    ", [m.name for m in dq.class_members])

charged leptons: ['e', 'mu', 'ta']
up quarks:       ['u', 'c', 't']
down quarks:     ['d', 's', 'b']


## 5. Gauge groups


In [5]:
U1Y = GaugeGroup("U1Y", abelian=True, coupling=g1, gauge_boson=B, ghost_field=ghB, charge="Y")
SU2L = GaugeGroup(
    "SU2L", abelian=False, coupling=g2, gauge_boson=Wi, ghost_field=ghWi,
    structure_constant=weak_structure_constant,
    representations=(GaugeRepresentation(index=WEAK_FUND_INDEX, generator_builder=weak_gauge_generator, name="doublet"),),
)
SU3C = GaugeGroup(
    "SU3C", abelian=False, coupling=g3, gauge_boson=G, ghost_field=ghG,
    structure_constant=structure_constant,
    representations=(GaugeRepresentation(index=COLOR_FUND_INDEX, generator_builder=gauge_generator, name="fundamental"),),
)
gauge_groups = (U1Y, SU2L, SU3C)
gauge_groups

(GaugeGroup(name='U1Y', abelian=True, coupling=Parameter(name='g1', symbol=g1, indices=(), complex_param=False, internal=True, value=None, components={}, allow_summation=None, unitary_partner=None), gauge_boson=Field(name='B', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=4, is_flavor=False, prefix='mu'),), kind='vector', statistics='boson', symbol=B, conjugate_symbol=None, mass=None, quantum_numbers={}, ghost_of=None, goldstone_of=None, flavor_index=None, class_members=()), ghost_field=Field(name='ghB', spin=0, self_conjugate=False, indices=(), kind='ghost', statistics='fermion', symbol=ghB, conjugate_symbol=ghBbar, mass=None, quantum_numbers={}, ghost_of=Field(name='B', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=4, is_flavor=False, prefix='mu'),), k

## 6. Lagrangian sectors


In [6]:
mu, nu = S("mu"), S("nu")
aW, aC = S("aW"), S("aC")
sp, ii, jj, cc = S("sp"), S("ii"), S("jj"), S("cc")
f1, f2, f3 = S("ff1"), S("ff2"), S("ff3")

# LGauge: -1/4 B.B - 1/4 Wi.Wi - 1/4 G.G
LGauge = (
    -num(1) / num(4) * FieldStrength(U1Y, mu, nu) * FieldStrength(U1Y, mu, nu)
    - num(1) / num(4) * FieldStrength(SU2L, mu, nu, aW) * FieldStrength(SU2L, mu, nu, aW)
    - num(1) / num(4) * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC)
)

# LFermions: i psibar gamma^mu D_mu psi
LFermions = (
    I * QL.bar * Gamma(mu) * CovD(QL, mu)
    + I * LL.bar * Gamma(mu) * CovD(LL, mu)
    + I * uR.bar * Gamma(mu) * CovD(uR, mu)
    + I * dR.bar * Gamma(mu) * CovD(dR, mu)
    + I * lR.bar * Gamma(mu) * CovD(lR, mu)
)

# LHiggs: |D Phi|^2 + muH^2 |Phi|^2 - lam |Phi|^4
LHiggs = (
    CovD(Phi.bar, mu) * CovD(Phi, mu)
    + lam * vev**2 * Phi.bar * Phi
    - lam * Phi.bar * Phi * Phi.bar * Phi
)

# LYukawa: down, lepton and up Yukawa terms + hermitian conjugates
LYukawa = (
    -Yd(f2, f3) * CKM(f1, f2) * QL.bar(sp, ii, f1, cc) * dR(sp, f3, cc) * Phi(ii)
    - Ye(f1, f3) * LL.bar(sp, ii, f1) * lR(sp, f3) * Phi(ii)
    - Yu(f1, f2) * QL.bar(sp, ii, f1, cc) * uR(sp, f2, cc) * Phi.bar(jj) * weak_eps2(ii, jj)
    - YdDag(f3, f2) * CKMDag(f2, f1) * Phi.bar(ii) * dR.bar(sp, f3, cc) * QL(sp, ii, f1, cc)
    - YeDag(f3, f1) * Phi.bar(ii) * lR.bar(sp, f3) * LL(sp, ii, f1)
    - YuDag(f2, f1) * weak_eps2(ii, jj) * Phi(jj) * uR.bar(sp, f2, cc) * QL(sp, ii, f1, cc)
)

# Manual R_xi gauge-fixing and ghost sectors
LGaugeFixing = (
    GaugeFixing(U1Y, xi=xiA.value)
    + GaugeFixing(SU2L, xi=xiW.value)
    + GaugeFixing(SU3C, xi=xiG.value)
)


def real_conjugate(value, *real_values):
    result = value.conj() if hasattr(value, "conj") else value
    for item in real_values:
        symbol = item.symbol if hasattr(item, "symbol") else item
        result = result.replace(symbol.conj(), symbol)
    return result


def electroweak_scalar_ghost_lagrangian():
    """Faddeev-Popov Higgs/Goldstone term in Feynman gauge."""
    zero = num(0)
    generators = (
        ((-I * g1.symbol * HALF, zero), (zero, -I * g1.symbol * HALF)),
        ((zero, -I * g2.symbol * HALF), (-I * g2.symbol * HALF, zero)),
        ((zero, -g2.symbol * HALF), (g2.symbol * HALF, zero)),
        ((-I * g2.symbol * HALF, zero), (zero, I * g2.symbol * HALF)),
    )
    vacuum = (zero, vev.symbol * INV_SQRT2)
    vacuum_images = tuple(
        tuple(
            sum((matrix[row][column] * vacuum[column] for column in range(2)), zero)
            for row in range(2)
        )
        for matrix in generators
    )
    ghosts = (ghB, ghWi(num(1)), ghWi(num(2)), ghWi(num(3)))
    antighosts = (ghB.bar, ghWi.bar(num(1)), ghWi.bar(num(2)), ghWi.bar(num(3)))

    result = zero
    for left in range(4):
        for right in range(4):
            for component in range(2):
                phi_coefficient = -sum(
                    (
                        real_conjugate(vacuum_images[left][row], g1, g2, vev)
                        * generators[right][row][component]
                        for row in range(2)
                    ),
                    zero,
                )
                phibar_coefficient = -sum(
                    (
                        real_conjugate(generators[right][row][component], g1, g2, vev)
                        * vacuum_images[left][row]
                        for row in range(2)
                    ),
                    zero,
                )
                if phi_coefficient.expand().to_canonical_string() != "0":
                    result += phi_coefficient * antighosts[left] * ghosts[right] * Phi(num(component + 1))
                if phibar_coefficient.expand().to_canonical_string() != "0":
                    result += phibar_coefficient * antighosts[left] * ghosts[right] * Phi.bar(num(component + 1))
    return result


LGhostScalar = electroweak_scalar_ghost_lagrangian()
LGhost = (
    PartialD(ghB.bar, mu) * PartialD(ghB, mu)
    + GhostLagrangian(SU2L)
    + GhostLagrangian(SU3C)
    + LGhostScalar
)

LSM = LGauge + LFermions + LHiggs + LYukawa + (LGaugeFixing + LGhost if FeynmanGauge else 0)
show("LGauge", LGauge)
show("LHiggs", LHiggs)
show("LGaugeFixing", LGaugeFixing)
show("LGhost", LGhost)

LGauge
-1/4 * FieldStrength(U1Y, mu, nu) * FieldStrength(U1Y, mu, nu) + -1/4 * FieldStrength(SU2L, mu, nu, aW) * FieldStrength(SU2L, mu, nu, aW) + -1/4 * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC)

LHiggs
CovD(Phi.bar, mu) * CovD(Phi, mu) + lam*vev^2 * Phi.bar * Phi + -lam * Phi.bar * Phi * Phi.bar * Phi

LGaugeFixing
GaugeFixing(U1Y, xi=xiA) + GaugeFixing(SU2L, xi=xiW) + GaugeFixing(SU3C, xi=xiG)

LGhost
PartialD(ghB.bar, mu) * PartialD(ghB, mu) + GhostLagrangian(SU2L) + GhostLagrangian(SU3C) + -1/4*(1/2)^(1/2)*g1^2*vev * ghB.bar * ghB * Phi + -1/4*(1/2)^(1/2)*g1^2*vev * ghB.bar * ghB * Phi.bar + -1/4*(1/2)^(1/2)*g1*g2*vev * ghB.bar * ghWi * Phi + -1/4*(1/2)^(1/2)*g1*g2*vev * ghB.bar * ghWi * Phi.bar + -1𝑖/4*(1/2)^(1/2)*g1*g2*vev * ghB.bar * ghWi * Phi + 1𝑖/4*(1/2)^(1/2)*g1*g2*vev * ghB.bar * ghWi * Phi.bar + 1/4*(1/2)^(1/2)*g1*g2*vev * ghB.bar * ghWi * Phi + 1/4*(1/2)^(1/2)*g1*g2*vev * ghB.bar * ghWi * Phi.bar + -1/4*(1/2)^(1/2)*g1*g2*vev * ghWi.bar * ghB * Phi

## 7. Field `Definitions` (symmetry breaking)

In `SM.fr`, the Lagrangian (section 6) uses gauge-basis fields (`B`, `Wi`, `Phi`,
`LL`, `QL`, ...). Section 7 is the **`Definitions`** block that rewrites them into
physical particles (`A`, `Z`, `W`, `H`, `e`, `u`, `d`, ...).

Each line is one `FieldTransformation(source, expr)`:

```python
FieldTransformation(B, -sw * Z + cw * A)              # boson mixing
FieldTransformation(Phi, vev*c + c*H + I*c*G0, components={0: 2})  # vev shift
FieldTransformation(LL, ProjM * l, components={1: 2})   # chiral projector
FieldTransformation(QL, rotation(CKM, CKMDag) * ProjM * dq, components={1: 2})  # + CKM
```

- **Bosons / Higgs / ghosts** — scalar-weighted sums of fields (and constants for the vev).
- **Fermions** — multiply by `ProjM` / `ProjP` and optionally `rotation(CKM, CKMDag)`.
  FeynPy wires spinor/flavor indices and conjugates automatically for `.bar` occurrences.

`components={slot: value}` restricts a rule to one multiplet component (e.g. `Wi[1]`).

In [7]:
swv, cwv = sw.value, cw.value
CKM_rot = rotation(CKM, CKMDag)

transformations = (
    # bosons
    FieldTransformation(B, -swv * Z + cwv * A),
    FieldTransformation(Wi, INV_SQRT2 * W.bar + INV_SQRT2 * W, components={1: 1}),
    FieldTransformation(Wi, INV_SQRT2 / I * W.bar - INV_SQRT2 / I * W, components={1: 2}),
    FieldTransformation(Wi, cwv * Z + swv * A, components={1: 3}),
    # Higgs / Goldstones
    FieldTransformation(Phi, -I * GP, components={0: 1}),
    FieldTransformation(Phi, vev.symbol * INV_SQRT2 + INV_SQRT2 * H + I * INV_SQRT2 * G0, components={0: 2}),
    # ghosts
    FieldTransformation(ghB, -swv * ghZ + cwv * ghA),
    FieldTransformation(ghWi, INV_SQRT2 * ghWp + INV_SQRT2 * ghWm, components={0: 1}),
    FieldTransformation(ghWi, -INV_SQRT2 / I * ghWp + INV_SQRT2 / I * ghWm, components={0: 2}),
    FieldTransformation(ghWi, cwv * ghZ + swv * ghA, components={0: 3}),
    # fermions  (SM.fr: LL -> ProjM l, QL -> CKM ProjM d, ...)
    FieldTransformation(LL, ProjM * vl, components={1: 1}),
    FieldTransformation(LL, ProjM * l,  components={1: 2}),
    FieldTransformation(lR, ProjP * l),
    FieldTransformation(QL, ProjM * uq, components={1: 1}),
    FieldTransformation(QL, CKM_rot * ProjM * dq, components={1: 2}),
    FieldTransformation(uR, ProjP * uq),
    FieldTransformation(dR, ProjP * dq),
)
print(len(transformations), "field definitions")

17 field definitions


### Weak-index unfolding (technical)

`Wi` and `Phi` carry a symbolic SU(2) index (`1`, `2`, or `3`). Before the
definitions can act on `Wi[1]`, `Wi[2]`, `Wi[3]`, FeynPy must expand that index
to explicit components (FeynRules: `ExpandIndices[..., FlavorExpand->SU2W]`).
The numeric Pauli/epsilon table below is written explicitly (no helper import).

In [8]:
def concrete(builder, *values):
    labels = tuple(S(f"component_{i}") for i in range(len(values)))
    expr = builder(*labels)
    for lbl, val in zip(labels, values):
        expr = expr.replace(lbl, num(val))
    return expr


weak_tensor_components = {}
pauli_over_two = {
    (1, 1, 1): 0, (1, 1, 2): HALF, (1, 2, 1): HALF, (1, 2, 2): 0,
    (2, 1, 1): 0, (2, 1, 2): -I * HALF, (2, 2, 1): I * HALF, (2, 2, 2): 0,
    (3, 1, 1): HALF, (3, 1, 2): 0, (3, 2, 1): 0, (3, 2, 2): -HALF,
}
for labels, value in pauli_over_two.items():
    weak_tensor_components[concrete(weak_gauge_generator, *labels)] = value
for a in range(1, 4):
    for b in range(1, 4):
        for c in range(1, 4):
            if len({a, b, c}) < 3:
                v = 0
            else:
                vals = (a, b, c)
                inv = sum(x > y for i, x in enumerate(vals) for y in vals[i + 1:])
                v = -1 if inv % 2 else 1
            weak_tensor_components[concrete(weak_structure_constant, a, b, c)] = num(v)
for a in range(1, 3):
    for b in range(1, 3):
        v = 1 if (a, b) == (1, 2) else -1 if (a, b) == (2, 1) else 0
        weak_tensor_components[concrete(weak_eps2, a, b)] = num(v)
print(len(weak_tensor_components), "SU(2) tensor components")

43 SU(2) tensor components


## 8. Build and extract Feynman rules

Three steps after writing `LSM` (automatic in FeynRules after loading `SM.fr`):

1. **Compile** the gauge-basis Lagrangian (`model.lagrangian()` expands `CovD` / `FieldStrength`).
2. **Unfold** weak indices so `Wi[1]` is an explicit slot (`expand_index_components`).
3. **Apply** the `Definitions` (`transform_fields(*transformations)`) to rewrite
   `B → A/Z`, `LL → ProjM l`, etc.

In [9]:
source_fields = (LL, lR, QL, uR, dR, Phi, B, Wi, G, ghB, ghWi, ghG)
source_model = Model(
    name="Standard Model (gauge basis)",
    gauge_groups=gauge_groups,
    fields=source_fields,
    parameters=all_parameters,
    lagrangian_decl=LSM,
)

real_symbols = (g1, g2, g3, lam, vev, MW, MZ, MH, sw.value, cw.value, ee.value)

L = (
    source_model.lagrangian()
    .expand_index_components(WEAK_FUND_INDEX, WEAK_ADJ_INDEX, tensor_components=weak_tensor_components)
    .transform_fields(*transformations, repeat=False, real_symbols=real_symbols)
    .simplify_parameter_identities()
)

# Physical-basis model used to ask for Feynman rules.
physical_fields = (vl, l, uq, dq, H, G0, GP, W, Z, A, G) + ((ghA, ghZ, ghWp, ghWm, ghG) if FeynmanGauge else ())
sm = Model(name=ModelName, fields=physical_fields, parameters=all_parameters)
sm._compiled_lagrangian = L

print("gauge-basis compiled terms:", len(source_model.lagrangian().terms))
print("broken-phase terms:        ", len(L.terms))
print("vertex signatures:         ", len(L.vertex_signatures()))

gauge-basis compiled terms: 94
broken-phase terms:         271
vertex signatures:          124


In [10]:
def compile_sector_piece(label, lagrangian_decl):
    source_piece = Model(
        name=f"Standard Model sector ({label})",
        gauge_groups=gauge_groups,
        fields=source_fields,
        parameters=all_parameters,
        lagrangian_decl=lagrangian_decl,
    )
    source_lagrangian = source_piece.lagrangian()
    broken_lagrangian = (
        source_lagrangian
        .expand_index_components(
            WEAK_FUND_INDEX,
            WEAK_ADJ_INDEX,
            tensor_components=weak_tensor_components,
        )
        .transform_fields(*transformations, repeat=False, real_symbols=real_symbols)
        .simplify_parameter_identities()
    )
    return source_lagrangian, broken_lagrangian


def signature_counts(compiled):
    signatures = compiled.vertex_signatures()
    return len(signatures), sum(signature.arity >= 3 for signature in signatures)


sector_defs = (
    ("Gauge", LGauge),
    ("Matter", LFermions),
    ("Higgs", LHiggs),
    ("Yukawa", LYukawa),
    ("Gauge fixing", LGaugeFixing),
    ("Ghost", LGhost),
)

print("vertex-signature counts by SM.fr sector")
header = f"{'sector':<14} {'source all':>10} {'source >=3':>11} {'broken all':>10} {'broken >=3':>11}"
print(header)
print("-" * len(header))
for label, lagrangian_decl in sector_defs:
    source_lagrangian, broken_lagrangian = compile_sector_piece(label, lagrangian_decl)
    source_all, source_interactions = signature_counts(source_lagrangian)
    broken_all, broken_interactions = signature_counts(broken_lagrangian)
    print(f"{label:<14} {source_all:>10} {source_interactions:>11} {broken_all:>10} {broken_interactions:>11}")


vertex-signature counts by SM.fr sector
sector         source all  source >=3 broken all  broken >=3
------------------------------------------------------------
Gauge                   7           4         12           8
Matter                 15          10         17          13
Higgs                   7           6         55          46
Yukawa                  6           6         13          10
Gauge fixing            3           0          5           0
Ghost                  13          10         31          24


In [11]:
higgs_model = Model(
    name="Higgs only",
    gauge_groups=gauge_groups,
    parameters=all_parameters,
    lagrangian_decl=LHiggs,
)

show_model("Higgs sector", higgs_model.feynman_rule())

Higgs sector
7 vertex signature(s)

Vertex: ('Phi.bar', 'Phi', 'B')
Rule: -1𝑖/2*g1*pcomp(q1,mu3)*g(cof(2,w1),cof(2,w2))+1𝑖/2*g1*pcomp(q2,mu3)*g(cof(2,w1),cof(2,w2))

Vertex: ('Phi.bar', 'Phi', 'B', 'B')
Rule: 1𝑖/2*g1^2*g(cof(2,w1),cof(2,w2))*g(mink(4,mu3),mink(4,mu4))

Vertex: ('Phi.bar', 'Phi', 'Wi')
Rule: -1𝑖*g2*pcomp(q1,mu3)*t(coad(3,aw3),cof(2,w1),cof(2,w2))+1𝑖*g2*pcomp(q2,mu3)*t(coad(3,aw3),cof(2,w1),cof(2,w2))

Vertex: ('Phi.bar', 'Phi', 'Wi', 'Wi')
Rule: 1𝑖*g2^2*t(coad(3,aw3),cof(2,w1),cof(2,w_mid_Phi_SU2L))*t(coad(3,aw4),cof(2,w_mid_Phi_SU2L),cof(2,w2))*g(mink(4,mu3),mink(4,mu4))+1𝑖*g2^2*t(coad(3,aw3),cof(2,w_mid_Phi_SU2L),cof(2,w2))*t(coad(3,aw4),cof(2,w1),cof(2,w_mid_Phi_SU2L))*g(mink(4,mu3),mink(4,mu4))

Vertex: ('Phi.bar', 'Phi', 'B', 'Wi')
Rule: 1𝑖*g1*g2*t(coad(3,aw4),cof(2,w1),cof(2,w2))*g(mink(4,mu3),mink(4,mu4))

Vertex: ('Phi.bar', 'Phi')
Rule: -1𝑖*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)*g(cof(2,w1),cof(2,w2))+1𝑖*lam*vev^2*g(cof(2,w1),cof(2,w2))

Vertex: ('Phi.bar', 'Phi',

In [12]:
Lh_higgs = higgs_model.lagrangian()

Lh_physical = Lh_higgs.transform_fields(
    *transformations,          # your FieldTransformation(...) tuple
    repeat=False,              # single simultaneous pass (SM.fr-style stage)
    real_symbols=real_symbols, # e.g. (vev.symbol,) or your tuple used in notebook
).simplify_parameter_identities()



In [13]:
# 1) compile the declaration piece
higgs_model = Model(
    name="Higgs only",
    gauge_groups=gauge_groups,
    parameters=all_parameters,
    lagrangian_decl=LHiggs,
)
Lh = higgs_model.lagrangian()

# 2) unfold both weak-doublet and weak-adjoint components before transformations
Lh = Lh.expand_index_components(
    WEAK_FUND_INDEX,
    WEAK_ADJ_INDEX,
    tensor_components=weak_tensor_components,
)

# 3) apply field transformations to physical fields
Lh_phys = (
    Lh.transform_fields(
        *transformations,
        repeat=False,
        real_symbols=real_symbols,
    )
    .simplify_parameter_identities()
)

all_higgs_rules_after_ssb = Lh_phys.feynman_rule()  # vacuum-energy constants are omitted
higgs_rules_after_ssb = {
    signature: rule
    for signature, rule in all_higgs_rules_after_ssb.items()
    if rule.expand().to_canonical_string() != "0"
}

print("all physical signatures:", len(all_higgs_rules_after_ssb))
print("non-zero rules:          ", len(higgs_rules_after_ssb))
show_model("Non-zero Higgs-sector Feynman rules after field transformations", higgs_rules_after_ssb)

all physical signatures: 55
non-zero rules:           46
Non-zero Higgs-sector Feynman rules after field transformations
46 vertex signature(s)

Vertex: ('GP.bar', 'GP', 'Z')
Rule: (g1^2+g2^2)^(-1/2)*-1𝑖/2*g1^2*pcomp(q2,mu3)+(g1^2+g2^2)^(-1/2)*-1𝑖/2*g2^2*pcomp(q1,mu3)+(g1^2+g2^2)^(-1/2)*1𝑖/2*g1^2*pcomp(q1,mu3)+(g1^2+g2^2)^(-1/2)*1𝑖/2*g2^2*pcomp(q2,mu3)

Vertex: ('GP.bar', 'GP', 'A')
Rule: (g1^2+g2^2)^(-1/2)*-1𝑖*g1*g2*pcomp(q1,mu3)+(g1^2+g2^2)^(-1/2)*1𝑖*g1*g2*pcomp(q2,mu3)

Vertex: ('G0', 'Z')
Rule: (g1^2+g2^2)^(-1/2)*1/2*g1^2*pcomp(q1,mu2)*vev+(g1^2+g2^2)^(-1/2)*1/2*g2^2*pcomp(q1,mu2)*vev

Vertex: ('H', 'G0', 'Z')
Rule: (g1^2+g2^2)^(-1/2)*-1/2*g1^2*pcomp(q1,mu3)+(g1^2+g2^2)^(-1/2)*-1/2*g2^2*pcomp(q1,mu3)+(g1^2+g2^2)^(-1/2)*1/2*g1^2*pcomp(q2,mu3)+(g1^2+g2^2)^(-1/2)*1/2*g2^2*pcomp(q2,mu3)

Vertex: ('GP.bar', 'GP', 'Z', 'Z')
Rule: (g1^2+g2^2)^(-1)*-1𝑖*g1^2*g2^2*g(mink(4,mu3),mink(4,mu4))+(g1^2+g2^2)^(-1)*1𝑖/2*g1^4*g(mink(4,mu3),mink(4,mu4))+(g1^2+g2^2)^(-1)*1𝑖/2*g2^4*g(mink(4,mu3),mink(4,

## 9. Mini-sector walkthrough: Yukawa sector

To see the transformation pipeline without the full Standard Model clutter, it
is cleaner to isolate just `LYukawa`. This one sector is already enough to show:

- weak-doublet index unfolding on `QL`, `LL`, `Phi`;
- the Higgs replacement `Phi -> vev + H + i G0` and `Phi[1] -> -i GP`;
- the chiral field definitions `LL, QL, lR, uR, dR -> vl, l, uq, dq`;
- the CKM rotation in the charged quark piece.

We keep the stages visible: source declaration, after weak-index unfolding, and
after the field transformations.

In [14]:
Lyukawa_demo = LYukawa

yukawa_demo_model = Model(
    name="SM Yukawa sector (gauge basis)",
    gauge_groups=gauge_groups,
    parameters=all_parameters,
    lagrangian_decl=Lyukawa_demo,
)

show("Before SSB: LYukawa", Lyukawa_demo)

Lyukawa_gauge = yukawa_demo_model.lagrangian()
Lyukawa_components = Lyukawa_gauge.expand_index_components(
    WEAK_FUND_INDEX,
    tensor_components=weak_tensor_components,
)
show_compiled("After weak-index unfolding", Lyukawa_components)

yukawa_rule_sources = {Phi, LL, lR, QL, uR, dR}
yukawa_rules = tuple(rule for rule in transformations if rule.source in yukawa_rule_sources)
Lyukawa_physical = (
    Lyukawa_components
    .transform_fields(*yukawa_rules, repeat=False, real_symbols=real_symbols)
    .simplify_parameter_identities()
)

yukawa_source_names = {"Phi", "LL", "QL", "lR", "uR", "dR"}
yukawa_survivors = sorted(
    {
        occ.field.name
        for term in Lyukawa_physical.terms
        for occ in term.fields
        if occ.field.name in yukawa_source_names
    }
)
show_compiled("After field transformations", Lyukawa_physical)
print("compiled gauge-basis terms:", len(Lyukawa_gauge.terms))
print("after weak unfolding:    ", len(Lyukawa_components.terms))
print("after transformations:   ", len(Lyukawa_physical.terms))
print("source-basis survivors:  ", yukawa_survivors)


Before SSB: LYukawa
-Yd(ff2,ff3)*CKM(ff1,ff2) * QL.bar * dR * Phi + -Ye(ff1,ff3) * LL.bar * lR * Phi + -weak_eps2(cof(2, ii),cof(2, jj))*Yu(ff1,ff2) * QL.bar * uR * Phi.bar + -YdDag(ff3,ff2)*CKMDag(ff2,ff1) * Phi.bar * dR.bar * QL + -YeDag(ff3,ff1) * Phi.bar * lR.bar * LL + -weak_eps2(cof(2, ii),cof(2, jj))*YuDag(ff2,ff1) * Phi * uR.bar * QL

After weak-index unfolding
-1*CKM(ff1,ff2)*Phi(1)*QLbar(sp,1,ff1,cc)*Yd(ff2,ff3)*dR(sp,ff3,cc)
-1*CKM(ff1,ff2)*Phi(2)*QLbar(sp,2,ff1,cc)*Yd(ff2,ff3)*dR(sp,ff3,cc)
-1*LLbar(sp,1,ff1)*Phi(1)*Ye(ff1,ff3)*lR(sp,ff3)
-1*LLbar(sp,2,ff1)*Phi(2)*Ye(ff1,ff3)*lR(sp,ff3)
-1*Phibar(2)*QLbar(sp,1,ff1,cc)*Yu(ff1,ff2)*uR(sp,ff2,cc)
Phibar(1)*QLbar(sp,2,ff1,cc)*Yu(ff1,ff2)*uR(sp,ff2,cc)
CKMDag(ff2,ff1)*Phibar(1)*QL(sp,1,ff1,cc)*YdDag(ff3,ff2)*dRbar(sp,ff3,cc)
CKMDag(ff2,ff1)*Phibar(2)*QL(sp,2,ff1,cc)*YdDag(ff3,ff2)*dRbar(sp,ff3,cc)
LL(sp,1,ff1)*Phibar(1)*YeDag(ff3,ff1)*lRbar(sp,ff3)
LL(sp,2,ff1)*Phibar(2)*YeDag(ff3,ff1)*lRbar(sp,ff3)
Phi(2)*QL(sp,1,ff1,cc)*YuDag(

In [15]:
def cf_sector(expr):
    return canonize_full(expr, infer_indices=True, field_heads=physical_fields, run_color=False)


show(
    "Gamma(dbar, d)   from the vev piece of Phi",
    cf_sector(Lyukawa_physical.feynman_rule(dq.bar, dq, simplify=True)),
)
show(
    "Gamma(dbar, d, H)   from the physical Higgs piece of Phi",
    cf_sector(Lyukawa_physical.feynman_rule(dq.bar, dq, H, simplify=True)),
)
show(
    "Gamma(ubar, d, GP)   charged Goldstone piece with CKM",
    cf_sector(Lyukawa_physical.feynman_rule(uq.bar, dq, GP, simplify=True)),
)


Gamma(dbar, d)   from the vev piece of Phi
(1/2)^(1/2)*-1𝑖*PL(i1,i2)*YdDag(fl1,fl2)*vev*g(cof(3,c1),cof(3,c2))+(1/2)^(1/2)*-1𝑖*PR(i1,i2)*Yd(fl1,fl2)*vev*g(cof(3,c1),cof(3,c2))

Gamma(dbar, d, H)   from the physical Higgs piece of Phi
(1/2)^(1/2)*-1𝑖*PL(i1,i2)*YdDag(fl1,fl2)*g(cof(3,c1),cof(3,c2))+(1/2)^(1/2)*-1𝑖*PR(i1,i2)*Yd(fl1,fl2)*g(cof(3,c1),cof(3,c2))

Gamma(ubar, d, GP)   charged Goldstone piece with CKM
-1*PR(i1,i2)*CKM(fl1,fl_canon_3)*Yd(fl_canon_3,fl2)*g(cof(3,c1),cof(3,c2))+PL(i1,i2)*CKM(fl_canon_4,fl2)*YuDag(fl1,fl_canon_4)*g(cof(3,c1),cof(3,c2))



## 10. Feynman rules

The physical-basis vertices, extracted exactly as in `list_lagrangians.ipynb`
with `L.feynman_rule(...)`.

In [16]:
def cf(expr):
    return canonize_full(expr, infer_indices=True, field_heads=tuple(sm.fields), run_color=False)


show("Gamma(W-, W+, A)", L.feynman_rule(W.bar, W, A, simplify=True))
show("Gamma(W-, W+, Z)", L.feynman_rule(W.bar, W, Z, simplify=True))
show("Gamma(W-, W+, A, A)", L.feynman_rule(W.bar, W, A, A, simplify=True))
show("Gamma(H, W-, W+)", L.feynman_rule(H, W.bar, W, simplify=True))
show("Gamma(H, Z, Z)", L.feynman_rule(H, Z, Z, simplify=True))
show("Gamma(H, H, H)", L.feynman_rule(H, H, H, simplify=True))
show("Gamma(lbar, l, A)  (lepton QED current)", cf(L.feynman_rule(l.bar, l, A, simplify=True)))
show("Gamma(ubar, d, W+)  (CKM charged current)", L.feynman_rule(uq.bar, dq, W, simplify=True))
show("Gamma(ubar, u, G)   (QCD vertex)", L.feynman_rule(uq.bar, uq, G, simplify=True))

Gamma(W-, W+, A)
(g1^2+g2^2)^(-1/2)*-1𝑖*g1*g2*pcomp(q1,mu2)*g(mink(4,mu1),mink(4,mu3))+(g1^2+g2^2)^(-1/2)*-1𝑖*g1*g2*pcomp(q2,mu3)*g(mink(4,mu1),mink(4,mu2))+(g1^2+g2^2)^(-1/2)*-1𝑖*g1*g2*pcomp(q3,mu1)*g(mink(4,mu2),mink(4,mu3))+(g1^2+g2^2)^(-1/2)*1𝑖*g1*g2*pcomp(q1,mu3)*g(mink(4,mu1),mink(4,mu2))+(g1^2+g2^2)^(-1/2)*1𝑖*g1*g2*pcomp(q2,mu1)*g(mink(4,mu2),mink(4,mu3))+(g1^2+g2^2)^(-1/2)*1𝑖*g1*g2*pcomp(q3,mu2)*g(mink(4,mu1),mink(4,mu3))

Gamma(W-, W+, Z)
(g1^2+g2^2)^(-1/2)*-1𝑖*g2^2*pcomp(q1,mu2)*g(mink(4,mu1),mink(4,mu3))+(g1^2+g2^2)^(-1/2)*-1𝑖*g2^2*pcomp(q2,mu3)*g(mink(4,mu1),mink(4,mu2))+(g1^2+g2^2)^(-1/2)*-1𝑖*g2^2*pcomp(q3,mu1)*g(mink(4,mu2),mink(4,mu3))+(g1^2+g2^2)^(-1/2)*1𝑖*g2^2*pcomp(q1,mu3)*g(mink(4,mu1),mink(4,mu2))+(g1^2+g2^2)^(-1/2)*1𝑖*g2^2*pcomp(q2,mu1)*g(mink(4,mu2),mink(4,mu3))+(g1^2+g2^2)^(-1/2)*1𝑖*g2^2*pcomp(q3,mu2)*g(mink(4,mu1),mink(4,mu3))

Gamma(W-, W+, A, A)
(g1^2+g2^2)^(-1)*-2𝑖*g1^2*g2^2*g(mink(4,mu1),mink(4,mu2))*g(mink(4,mu3),mink(4,mu4))+(g1^2+g2^2)^(-1)*1𝑖*g1^2*g2^2*g

## 11. Sanity checks (fully standalone)

No `build_standard_model()` is used below. We perform internal checks that the
manual construction worked:

- no source-basis fields (`B`, `Wi`, `Phi`, `LL`, `QL`, ...) survive in `L`;
- representative EW/QCD/ghost vertices are non-zero;
- gauge-fixing sector contributes longitudinal terms.

In [17]:
def canon(expr):
    return expr.expand().to_canonical_string()


source_basis_names = {"B", "Wi", "Phi", "LL", "QL", "lR", "uR", "dR", "ghB", "ghWi"}
survivors = sorted({occ.field.name for term in L.terms for occ in term.fields if occ.field.name in source_basis_names})
print("source-basis survivors:", survivors)

checks = {
    "WWA": L.feynman_rule(W.bar, W, A, simplify=True),
    "HWW": L.feynman_rule(H, W.bar, W, simplify=True),
    "llA": cf(L.feynman_rule(l.bar, l, A, simplify=True)),
    "udW": L.feynman_rule(uq.bar, dq, W, simplify=True),
    "uuG": L.feynman_rule(uq.bar, uq, G, simplify=True),
    "ghWp_kin": L.feynman_rule(ghWp.bar, ghWp, simplify=True),
    "ghWpWghA": L.feynman_rule(ghWp.bar, W, ghA, simplify=True),
    "AA_2pt": L.feynman_rule(A, A, simplify=True),
}

for key, expr in checks.items():
    print(f"{key:9} nonzero:", canon(expr) != "0")

source-basis survivors: []
WWA       nonzero: True
HWW       nonzero: True
llA       nonzero: True
udW       nonzero: True
uuG       nonzero: True
ghWp_kin  nonzero: True
ghWpWghA  nonzero: True
AA_2pt    nonzero: True


## 12. Exact FeynPy / FeynRules comparison: gauge sector

We now compare the complete physical pure-gauge sector against the exported
FeynRules rules in `sandbox/wolframnotebook/gauge_vertices_FeynRules.json`.
This is a symbolic comparison, not a display-string comparison:

1. signatures are matched as field multisets, while each rule is extracted in
   the exact FeynRules external-leg order;
2. FeynRules `ME`, `FV`, and `f` objects are parsed into the same native
   Symbolica/Spenso tensors used by FeynPy;
3. comparisons are made in the FeynRules-native electroweak basis;
4. Lorentz and color indices, including generated contracted indices, are
   canonicalized independently on both sides;
5. equality means that the canonical symbolic difference is exactly zero.

The adapter and report types live in `feynrules.comparison`, so later model and
sector comparisons can reuse the alignment and reporting layer while adding
parsers for further FeynRules tensor heads.

In [18]:
from symbolica import AtomType

from feynrules.comparison import (
    compare_feynrules_gauge_vertices,
    load_feynrules_json,
)


feynrules_gauge_path = ROOT / "sandbox" / "wolframnotebook" / "gauge_vertices_FeynRules.json"
feynrules_gauge_vertices = load_feynrules_json(feynrules_gauge_path)

# Isolate LGauge before comparing coverage: matter, Higgs, gauge-fixing, and
# ghost vertices must not affect the pure-gauge signature inventory.
_, L_gauge_physical = compile_sector_piece("Gauge comparison", LGauge)

gauge_comparison = compare_feynrules_gauge_vertices(
    L_gauge_physical,
    feynrules_gauge_vertices,
    field_map={"A": A, "W": W, "Wbar": W.bar, "Z": Z, "G": G},
    feynpy_name_aliases={"W.bar": "Wbar"},
)

print("reference file:", feynrules_gauge_path.relative_to(ROOT))
print("FeynRules vertices:", len(feynrules_gauge_vertices))
print("FeynPy gauge interactions (arity >= 3):", sum(s.arity >= 3 for s in L_gauge_physical.vertex_signatures()))

reference file: sandbox/wolframnotebook/gauge_vertices_FeynRules.json
FeynRules vertices: 8
FeynPy gauge interactions (arity >= 3): 8


In [19]:
def additive_term_count(expr):
    if expr is None:
        return 0
    expanded = expr.expand()
    return len(tuple(expanded)) if expanded.get_type() == AtomType.Add else 1


header = f"{'key':<18} {'legs':>4} {'FeynPy terms':>13} {'FR terms':>9} {'status':>10}"
print(header)
print("-" * len(header))
for row in gauge_comparison.rows:
    print(
        f"{row.reference.key:<18} {len(row.reference.fields):>4} "
        f"{additive_term_count(row.feynpy_rule):>13} "
        f"{additive_term_count(row.feynrules_rule):>9} "
        f"{row.status:>10}"
    )

print()
print("only in FeynRules:", gauge_comparison.feynrules_only)
print("only in FeynPy:   ", gauge_comparison.feynpy_only)
print(f"exact matches: {gauge_comparison.matched}/{len(gauge_comparison.rows)}")

assert gauge_comparison.all_match

key                legs  FeynPy terms  FR terms     status
----------------------------------------------------------
A|A|W|Wbar            4             3         3      MATCH
A|W|Wbar              3             6         6      MATCH
A|W|Wbar|Z            4             3         3      MATCH
G|G|G                 3             6         6      MATCH
G|G|G|G               4             6         6      MATCH
W|Wbar|Z              3             6         6      MATCH
W|Wbar|Z|Z            4             3         3      MATCH
W|W|Wbar|Wbar         4             3         3      MATCH

only in FeynRules: ()
only in FeynPy:    ()
exact matches: 8/8


### Hardest case: the quartic gluon vertex

The four-gluon rule is the strongest check in this export: it contains six
terms, products of two antisymmetric structure constants, three Lorentz
channels, and a contracted color-adjoint index whose generated name differs
between the two systems. The comparison below shows the two independently
canonicalized rules and their exact difference.

In [20]:
four_gluon = next(
    row for row in gauge_comparison.rows
    if row.reference.key == "G|G|G|G"
)

show("FeynPy canonical GGGG", four_gluon.feynpy_rule)
show("FeynRules canonical GGGG", four_gluon.feynrules_rule)
show("canonical difference", four_gluon.difference)

FeynPy canonical GGGG
-1𝑖*g3^2*f(coad(8,canon_dummy_1_1),coad(8,a1),coad(8,a2))*f(coad(8,canon_dummy_1_1),coad(8,a3),coad(8,a4))*g(mink(4,mu1),mink(4,mu3))*g(mink(4,mu2),mink(4,mu4))+-1𝑖*g3^2*f(coad(8,canon_dummy_1_1),coad(8,a1),coad(8,a3))*f(coad(8,canon_dummy_1_1),coad(8,a2),coad(8,a4))*g(mink(4,mu1),mink(4,mu2))*g(mink(4,mu3),mink(4,mu4))+-1𝑖*g3^2*f(coad(8,canon_dummy_1_1),coad(8,a1),coad(8,a4))*f(coad(8,canon_dummy_1_1),coad(8,a2),coad(8,a3))*g(mink(4,mu1),mink(4,mu2))*g(mink(4,mu3),mink(4,mu4))+1𝑖*g3^2*f(coad(8,canon_dummy_1_1),coad(8,a1),coad(8,a2))*f(coad(8,canon_dummy_1_1),coad(8,a3),coad(8,a4))*g(mink(4,mu1),mink(4,mu4))*g(mink(4,mu2),mink(4,mu3))+1𝑖*g3^2*f(coad(8,canon_dummy_1_1),coad(8,a1),coad(8,a3))*f(coad(8,canon_dummy_1_1),coad(8,a2),coad(8,a4))*g(mink(4,mu1),mink(4,mu4))*g(mink(4,mu2),mink(4,mu3))+1𝑖*g3^2*f(coad(8,canon_dummy_1_1),coad(8,a1),coad(8,a4))*f(coad(8,canon_dummy_1_1),coad(8,a2),coad(8,a3))*g(mink(4,mu1),mink(4,mu3))*g(mink(4,mu2),mink(4,mu4))

FeynRules cano

All eight interaction signatures and all eight symbolic rules match exactly.
This establishes the comparison pattern for subsequent sectors. Extending it
to matter, Higgs, Yukawa, gauge-fixing, and ghost outputs requires additional
FeynRules input adapters for tensors such as `IndexDelta`, `Ga`, generators,
projectors, flavor matrices, and ghost conventions; the signature alignment,
leg-order handling, canonicalization, zero-difference test, and report format
remain unchanged.

## 13. Exact FeynPy / FeynRules comparison: matter sector

The matter export contains 39 flavor-expanded fermion-gauge vertices. Besides
signature and parameter alignment, this comparison must reconcile two
different presentations of the Dirac structure:

- FeynRules uses `Ga` and compact products such as
  `TensDot[Ga, ProjM]` / `TensDot[Ga, ProjP]`;
- FeynPy currently returns the equivalent expanded gamma/gamma5 chains after
  applying the field transformations.

Both forms are parsed into native spinor tensors and reduced to the same
vector/axial-current basis using the Clifford identities
`gamma5 gamma = -gamma gamma5` and
`gamma5 gamma gamma5 = -gamma`. Color deltas and generators are then
canonicalized before taking the exact symbolic difference.

The supplied FeynRules file uses an identity CKM matrix. The notebook keeps a
Cabibbo rotation symbolic, so for this comparison only we explicitly evaluate
`cos(cabi) = 1` and `sin(cabi) = 0`. This also removes the four off-diagonal
Cabibbo vertices that are absent from the reference export.

In [21]:
from feynrules.comparison import compare_feynrules_matter_vertices


feynrules_matter_path = ROOT / "sandbox" / "wolframnotebook" / "matter_vertices_FeynRules.json"
feynrules_matter_vertices = load_feynrules_json(feynrules_matter_path)
_, L_matter_physical = compile_sector_piece("Matter comparison", LFermions)

matter_field_map = {"A": A, "Z": Z, "W": W, "Wbar": W.bar, "G": G}
matter_name_aliases = {"W.bar": "Wbar"}
for field_class in (vl, l, uq, dq):
    for member in field_class.class_members:
        matter_field_map[member.name] = member
        matter_field_map[f"{member.name}bar"] = member.bar
        matter_name_aliases[f"{member.name}.bar"] = f"{member.name}bar"

matter_comparison = compare_feynrules_matter_vertices(
    L_matter_physical,
    feynrules_matter_vertices,
    field_map=matter_field_map,
    feynpy_substitutions={cos: num(1), sin: num(0)},
    feynpy_name_aliases=matter_name_aliases,
)

print("reference file:", feynrules_matter_path.relative_to(ROOT))
print("FeynRules vertices:", len(feynrules_matter_vertices))
print("comparison convention: CKM = identity")

reference file: sandbox/wolframnotebook/matter_vertices_FeynRules.json
FeynRules vertices: 39
comparison convention: CKM = identity


In [22]:
header = f"{'key':<20} {'FeynPy terms':>13} {'FR terms':>9} {'status':>10}"
print(header)
print("-" * len(header))
for row in matter_comparison.rows:
    print(
        f"{row.reference.key:<20} "
        f"{additive_term_count(row.feynpy_rule):>13} "
        f"{additive_term_count(row.feynrules_rule):>9} "
        f"{row.status:>10}"
    )

print()
print("only in FeynRules:", matter_comparison.feynrules_only)
print("only in FeynPy:   ", matter_comparison.feynpy_only)
print(f"exact matches: {matter_comparison.matched}/{len(matter_comparison.rows)}")

assert matter_comparison.all_match

key                   FeynPy terms  FR terms     status
-------------------------------------------------------
A|b|bbar                         1         1      MATCH
A|c|cbar                         1         1      MATCH
A|d|dbar                         1         1      MATCH
A|e|ebar                         1         1      MATCH
A|mu|mubar                       1         1      MATCH
A|s|sbar                         1         1      MATCH
A|ta|tabar                       1         1      MATCH
A|t|tbar                         1         1      MATCH
A|u|ubar                         1         1      MATCH
bbar|t|Wbar                      2         2      MATCH
b|bbar|G                         1         1      MATCH
b|bbar|Z                         4         4      MATCH
b|tbar|W                         2         2      MATCH
cbar|s|W                         2         2      MATCH
c|cbar|G                         1         1      MATCH
c|cbar|Z                         4         4    

### Representative chiral checks

The neutral-current rule `bbar b Z` contains independent left- and
right-chiral couplings, while `dbar u Wbar` is a purely left-chiral charged
current. Their canonical differences test the projector reduction more
strongly than a vector-like photon or gluon vertex.

In [23]:
for key in ("b|bbar|Z", "dbar|u|Wbar"):
    row = next(item for item in matter_comparison.rows if item.reference.key == key)
    show(f"FeynPy canonical {key}", row.feynpy_rule)
    show(f"FeynRules canonical {key}", row.feynrules_rule)
    show(f"canonical difference {key}", row.difference)

FeynPy canonical b|bbar|Z
(g1^2+g2^2)^(-1/2)*-1𝑖/4*DiracVector(mu3,i1,i2)*g2^2*g(cof(3,c1),cof(3,c2))+(g1^2+g2^2)^(-1/2)*1𝑖/12*DiracVector(mu3,i1,i2)*g1^2*g(cof(3,c1),cof(3,c2))+(g1^2+g2^2)^(-1/2)*1𝑖/4*DiracAxial(mu3,i1,i2)*g1^2*g(cof(3,c1),cof(3,c2))+(g1^2+g2^2)^(-1/2)*1𝑖/4*DiracAxial(mu3,i1,i2)*g2^2*g(cof(3,c1),cof(3,c2))

FeynRules canonical b|bbar|Z
(g1^2+g2^2)^(-1/2)*-1𝑖/4*DiracVector(mu3,i1,i2)*g2^2*g(cof(3,c1),cof(3,c2))+(g1^2+g2^2)^(-1/2)*1𝑖/12*DiracVector(mu3,i1,i2)*g1^2*g(cof(3,c1),cof(3,c2))+(g1^2+g2^2)^(-1/2)*1𝑖/4*DiracAxial(mu3,i1,i2)*g1^2*g(cof(3,c1),cof(3,c2))+(g1^2+g2^2)^(-1/2)*1𝑖/4*DiracAxial(mu3,i1,i2)*g2^2*g(cof(3,c1),cof(3,c2))

canonical difference b|bbar|Z
0

FeynPy canonical dbar|u|Wbar
(1/2)^(1/2)*-1𝑖/2*DiracAxial(mu3,i1,i2)*g2*g(cof(3,c1),cof(3,c2))+(1/2)^(1/2)*1𝑖/2*DiracVector(mu3,i1,i2)*g2*g(cof(3,c1),cof(3,c2))

FeynRules canonical dbar|u|Wbar
(1/2)^(1/2)*-1𝑖/2*DiracAxial(mu3,i1,i2)*g2*g(cof(3,c1),cof(3,c2))+(1/2)^(1/2)*1𝑖/2*DiracVector(mu3,i1,i2)*g2*g(cof(3

All 39 matter-sector signatures and symbolic rules match exactly in the
identity-CKM convention. The checks include photon, gluon, neutral-current,
charged-current, color-delta, color-generator, vector, axial, left-chiral,
and right-chiral structures.

## 14. Exact FeynPy / FeynRules comparison: Yukawa sector

The Yukawa reference contains ten representative third-generation vertices:
neutral Higgs and Goldstone couplings plus charged-Goldstone quark and lepton
couplings. Compact `ProjM` / `ProjP` expressions and FeynPy's spinor tensors
are reduced to a common scalar/pseudoscalar bilinear basis before subtraction.

As in the matter comparison, the reference uses an identity CKM matrix. Its
diagonal Yukawa components are treated as real, matching the notebook's
`yu3`, `yd3`, and `ye3` component declarations.

In [24]:
from feynrules.comparison import (
    compare_feynrules_bosonic_vertices,
    compare_feynrules_yukawa_vertices,
)


feynrules_yukawa_path = ROOT / "sandbox" / "wolframnotebook" / "yukawa_vertices_FeynRules.json"
feynrules_yukawa_vertices = load_feynrules_json(feynrules_yukawa_path)
_, L_yukawa_physical = compile_sector_piece("Yukawa comparison", LYukawa)

yukawa_comparison = compare_feynrules_yukawa_vertices(
    L_yukawa_physical,
    feynrules_yukawa_vertices,
    field_map=matter_field_map | {"H": H, "G0": G0, "GP": GP, "GPbar": GP.bar},
    diagonal_yukawa_names={"yl": "ye"},
    feynpy_substitutions={cos: num(1), sin: num(0)},
    feynpy_name_aliases=matter_name_aliases | {"GP.bar": "GPbar"},
)

print("reference file:", feynrules_yukawa_path.relative_to(ROOT))
print("FeynRules vertices:", len(feynrules_yukawa_vertices))
print("comparison convention: real diagonal Yukawas, CKM = identity")

reference file: sandbox/wolframnotebook/yukawa_vertices_FeynRules.json
FeynRules vertices: 10
comparison convention: real diagonal Yukawas, CKM = identity


In [25]:
for row in yukawa_comparison.rows:
    print(f"{row.reference.key:<20} {row.status:>10}")
print()
print("only in FeynRules:", yukawa_comparison.feynrules_only)
print("only in FeynPy:   ", yukawa_comparison.feynpy_only)
print(f"exact matches: {yukawa_comparison.matched}/{len(yukawa_comparison.rows)}")

assert yukawa_comparison.all_match

bbar|GPbar|t              MATCH
b|bbar|G0                 MATCH
b|bbar|H                  MATCH
b|GP|tbar                 MATCH
G0|ta|tabar               MATCH
G0|t|tbar                 MATCH
GPbar|tabar|vt            MATCH
GP|ta|vtbar               MATCH
H|ta|tabar                MATCH
H|t|tbar                  MATCH

only in FeynRules: ()
only in FeynPy:    ()
exact matches: 10/10


## 15. Exact FeynPy / FeynRules comparison: Higgs sector

The Higgs export contains 38 non-zero cubic and quartic interactions from the
Higgs kinetic term and potential. The comparison covers derivative momenta,
Lorentz metrics, gauge couplings, the vacuum expectation value, and all
scalar combinatorial factors.

In [26]:
feynrules_higgs_path = ROOT / "sandbox" / "wolframnotebook" / "higgs_vertices_FeynRules.json"
feynrules_higgs_vertices = load_feynrules_json(feynrules_higgs_path)
_, L_higgs_physical = compile_sector_piece("Higgs comparison", LHiggs)

bosonic_field_map = {
    "A": A, "Z": Z, "W": W, "Wbar": W.bar,
    "H": H, "G0": G0, "GP": GP, "GPbar": GP.bar,
}
bosonic_name_aliases = {"W.bar": "Wbar", "GP.bar": "GPbar"}

higgs_comparison = compare_feynrules_bosonic_vertices(
    L_higgs_physical,
    feynrules_higgs_vertices,
    field_map=bosonic_field_map,
    feynpy_name_aliases=bosonic_name_aliases,
    minimum_scalar_fields=1,
    scalar_relations=("cw**2 + sw**2 - 1",),
)

print("reference file:", feynrules_higgs_path.relative_to(ROOT))
print("FeynRules vertices:", len(feynrules_higgs_vertices))

reference file: sandbox/wolframnotebook/higgs_vertices_FeynRules.json
FeynRules vertices: 38


In [27]:
for row in higgs_comparison.rows:
    print(f"{row.reference.key:<22} {row.status:>10}")
print()
print("only in FeynRules:", higgs_comparison.feynrules_only)
print("only in FeynPy:   ", higgs_comparison.feynpy_only)
print(f"exact matches: {higgs_comparison.matched}/{len(higgs_comparison.rows)}")

assert higgs_comparison.all_match

A|GPbar|W                   MATCH
A|GP|GPbar                  MATCH
A|GP|Wbar                   MATCH
G0|G0|H                     MATCH
G0|GPbar|W                  MATCH
G0|GP|Wbar                  MATCH
G0|H|Z                      MATCH
GPbar|H|W                   MATCH
GPbar|W|Z                   MATCH
GP|GPbar|H                  MATCH
GP|GPbar|Z                  MATCH
GP|H|Wbar                   MATCH
GP|Wbar|Z                   MATCH
H|H|H                       MATCH
H|W|Wbar                    MATCH
H|Z|Z                       MATCH
A|A|GP|GPbar                MATCH
A|G0|GPbar|W                MATCH
A|G0|GP|Wbar                MATCH
A|GPbar|H|W                 MATCH
A|GP|GPbar|Z                MATCH
A|GP|H|Wbar                 MATCH
G0|G0|G0|G0                 MATCH
G0|G0|GP|GPbar              MATCH
G0|G0|H|H                   MATCH
G0|G0|W|Wbar                MATCH
G0|G0|Z|Z                   MATCH
G0|GPbar|W|Z                MATCH
G0|GP|Wbar|Z                MATCH
GPbar|H|W|Z   

## 16. Exact FeynPy / FeynRules comparison: ghost sector

The ghost export contains 24 interactions. The notebook now includes both
parts of the Faddeev–Popov sector:

- derivative ghost–gauge interactions from `GhostLagrangian(...)`;
- the electroweak scalar–ghost term generated by the Higgs-dependent
  Feynman-gauge fixing functions.

For derivative ghost rules, FeynPy and FeynRules can place the derivative on
different external ghost legs. Their expressions are compared after applying
the exact all-incoming momentum identity `q1 + q2 + q3 = 0`. Scalar
electroweak coefficient identities are reduced exactly as rational symbolic
functions before the zero-difference test.

In [28]:
feynrules_ghost_path = ROOT / "sandbox" / "wolframnotebook" / "ghost_vertices_FeynRules.json"
feynrules_ghost_vertices = load_feynrules_json(feynrules_ghost_path)
_, L_ghost_physical = compile_sector_piece("Ghost comparison", LGhost)

ghost_field_map = bosonic_field_map | {
    "G": G,
    "ghA": ghA, "ghAbar": ghA.bar,
    "ghZ": ghZ, "ghZbar": ghZ.bar,
    "ghWp": ghWp, "ghWpbar": ghWp.bar,
    "ghWm": ghWm, "ghWmbar": ghWm.bar,
    "ghG": ghG, "ghGbar": ghG.bar,
}
ghost_name_aliases = bosonic_name_aliases | {
    "ghA.bar": "ghAbar", "ghZ.bar": "ghZbar",
    "ghWp.bar": "ghWpbar", "ghWm.bar": "ghWmbar",
    "ghG.bar": "ghGbar",
}

ghost_comparison = compare_feynrules_bosonic_vertices(
    L_ghost_physical,
    feynrules_ghost_vertices,
    field_map=ghost_field_map,
    feynpy_name_aliases=ghost_name_aliases,
    minimum_ghost_fields=2,
    use_momentum_conservation=True,
    scalar_relations=("cw**2 + sw**2 - 1",),
)

print("reference file:", feynrules_ghost_path.relative_to(ROOT))
print("FeynRules vertices:", len(feynrules_ghost_vertices))

reference file: sandbox/wolframnotebook/ghost_vertices_FeynRules.json
FeynRules vertices: 24


In [29]:
for row in ghost_comparison.rows:
    print(f"{row.reference.key:<24} {row.status:>10}")
print()
print("only in FeynRules:", ghost_comparison.feynrules_only)
print("only in FeynPy:   ", ghost_comparison.feynpy_only)
print(f"exact matches: {ghost_comparison.matched}/{len(ghost_comparison.rows)}")

assert ghost_comparison.all_match

A|ghWm|ghWmbar                MATCH
A|ghWp|ghWpbar                MATCH
G0|ghWm|ghWmbar               MATCH
G0|ghWp|ghWpbar               MATCH
G|ghG|ghGbar                  MATCH
ghAbar|ghWm|W                 MATCH
ghAbar|ghWp|Wbar              MATCH
ghA|ghWmbar|GPbar             MATCH
ghA|ghWmbar|Wbar              MATCH
ghA|ghWpbar|GP                MATCH
ghA|ghWpbar|W                 MATCH
ghWmbar|ghZ|GPbar             MATCH
ghWmbar|ghZ|Wbar              MATCH
ghWm|ghWmbar|H                MATCH
ghWm|ghWmbar|Z                MATCH
ghWm|ghZbar|GP                MATCH
ghWm|ghZbar|W                 MATCH
ghWpbar|ghZ|GP                MATCH
ghWpbar|ghZ|W                 MATCH
ghWp|ghWpbar|H                MATCH
ghWp|ghWpbar|Z                MATCH
ghWp|ghZbar|GPbar             MATCH
ghWp|ghZbar|Wbar              MATCH
ghZ|ghZbar|H                  MATCH

only in FeynRules: ()
only in FeynPy:    ()
exact matches: 24/24


The strict comparison now covers the complete supplied reference set:

- gauge: 8/8;
- matter: 39/39;
- Yukawa: 10/10;
- Higgs: 38/38;
- ghost: 24/24.

Every reported match is an exact symbolic zero-difference check after the
explicitly documented basis and convention transformations.

## Summary

This notebook defines the Standard Model the way a FeynPy user is meant to:

- one `Field(...)` per particle class (gauge basis + physical), one
  `GaugeGroup(...)` per gauge factor, one `Parameter(...)` per parameter;
- the Lagrangian written in the unbroken gauge basis as `LGauge + LFermions +
  LHiggs + LYukawa`, using only `FieldStrength`, `CovD`, `Gamma` and the Yukawa
  contractions;
- the spontaneous symmetry breaking expressed declaratively as the field
  `Definitions` (`FieldTransformation`s), exactly mirroring `SM.fr`.

The engine then unfolds the weak indices, applies the definitions and returns the
physical Feynman rules, which match the FeynRules `SM.fr` reference (last cell).

This notebook is now fully standalone: it manually defines

- `LGaugeFixing` via `GaugeFixing(...)` declarations;
- `LGhost` via an explicit abelian ghost kinetic term plus `GhostLagrangian(...)`
  for the non-abelian groups;
- ghost field transformations (`ghB`, `ghWi`) alongside boson/Higgs/fermion
  transformations.

So the full workflow is pure `feynpy` + explicit notebook declarations, without
calling `build_standard_model()`. 